# Ecuación de calor 1D: cuaderno guiado con verificaciones

Este cuaderno está diseñado para trabajar de forma gradual. La meta es implementar el método, comprobar su validez numérica y discutir su comportamiento.

Estudiaremos:

$$
\frac{\partial u}{\partial t}=\alpha\frac{\partial^2u}{\partial x^2},\quad x\in(0,1),\ t\in(0,T)
$$

con fronteras homogéneas y condición inicial:

$$u(0,t)=u(1,t)=0,\qquad u(x,0)=\sin(\pi x).$$

**Nota de notación temporal**: en el código, `t_final` representa el valor de `T` que fijamos para la simulación. El estado estacionario se interpreta como el límite cuando $t\to\infty$.


## 1) Contexto físico

La ecuación del calor describe un proceso de difusión: la energía térmica fluye desde regiones de mayor temperatura hacia regiones de menor temperatura.

- Cuando el gradiente es grande, el flujo térmico es más intenso.
- Con fronteras fijas en cero, la solución tiende a decaer en todo el dominio.
- Este modelo permite vincular interpretación física e implementación numérica.

## 2) Discretización FTCS

Usaremos una malla uniforme:

$$x_i=i\,\Delta x,\quad t^n=n\,\Delta t$$

Evaluamos la EDP en $(x_i,t^n)$ y aproximamos cada derivada:

1. Derivada temporal (adelante en tiempo):

$$
\left.\frac{\partial u}{\partial t}\right|_i^n\approx\frac{u_i^{n+1}-u_i^n}{\Delta t}
$$

2. Derivada espacial segunda (centrada):

$$
\left.\frac{\partial^2 u}{\partial x^2}\right|_i^n\approx\frac{u_{i+1}^n-2u_i^n+u_{i-1}^n}{\Delta x^2}
$$

Sustituyendo en la ecuación de calor:

$$
\frac{u_i^{n+1}-u_i^n}{\Delta t}=\alpha\frac{u_{i+1}^n-2u_i^n+u_{i-1}^n}{\Delta x^2}.
$$

Despejando, se obtiene el esquema explícito FTCS:

$$u_i^{n+1}=u_i^n+r\left(u_{i+1}^n-2u_i^n+u_{i-1}^n\right),\quad r=\frac{\alpha\Delta t}{\Delta x^2}.$$

En esta práctica revisaremos estabilidad con la condición $r\leq 1/2$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

In [ ]:
# Parámetros base
alpha = 1.0
L = 1.0
T = 0.12
N = 80

dx = L / (N + 1)
dt = 0.45 * dx**2 / alpha
r = alpha * dt / dx**2
print(f"dx={dx:.6f}, dt={dt:.8f}, r={r:.3f}")

## 3) Funciones de apoyo (solución exacta e inicial)

In [ ]:
def exact_solution(x, t, alpha=1.0):
    return np.exp(-alpha*np.pi**2*t) * np.sin(np.pi*x)

def initial_condition(x):
    return np.sin(np.pi*x)

## 4) TO DO A: construir la malla interior y el número de pasos de tiempo

In [ ]:
def build_grids(L, T, N, dt):
    """
    Regresa:
      x : nodos interiores (N nodos)
      nt: entero de pasos de tiempo para llegar a T (o superarlo ligeramente)
    """
    # TO DO
    # dx = ...
    # x = ...
    # nt = ...
    raise NotImplementedError("Completa build_grids")

## 5) TO DO B: implementar un paso FTCS con fronteras homogéneas

In [ ]:
def ftcs_step(u, r):
    """
    u: vector interior en tiempo n
    r: numero de difusión
    """
    # TO DO
    # unew = np.empty_like(u)
    # extremo izquierdo usa frontera u(0,t)=0
    # extremo derecho usa frontera u(1,t)=0
    # interior con slicing
    raise NotImplementedError("Completa ftcs_step")

## 6) TO DO C: completar el solver y registrar historial temporal

In [ ]:
def solve_heat_ftcs(alpha=1.0, L=1.0, T=0.12, N=80, dt=None, store_every=6):
    if dt is None:
        dx_local = L / (N + 1)
        dt = 0.45 * dx_local**2 / alpha

    x, nt = build_grids(L, T, N, dt)
    dx_local = L / (N + 1)
    r_local = alpha * dt / dx_local**2

    if r_local > 0.5:
        raise ValueError(f"Esquema inestable: r={r_local:.4f} > 0.5")

    u = initial_condition(x)
    U_hist = [u.copy()]
    t_hist = [0.0]

    # TO DO: ciclo temporal usando ftcs_step
    # for n in range(1, nt+1):
    #     ...

    # retorna último estado y también historial
    # return x, u, t_final, r_local, np.array(t_hist), np.array(U_hist)
    raise NotImplementedError("Completa solve_heat_ftcs")

## 7) Verificaciones automáticas

En esta sección validamos tres puntos:

- Parámetro de estabilidad dentro del rango esperado.
- Condiciones de frontera respetadas en todos los pasos.
- Error numérico frente a la solución analítica en `t_final`.

Recuerda: `t_final` es el horizonte temporal de la simulación; no equivale por definición al estado estacionario.


In [ ]:
x, u_num, t_final, r, t_hist, U_hist = solve_heat_ftcs(alpha=alpha, L=L, T=T, N=N, dt=dt)

assert isinstance(x, np.ndarray) and isinstance(u_num, np.ndarray)
assert x.ndim == 1 and u_num.ndim == 1
assert len(x) == N and len(u_num) == N
assert r <= 0.5 + 1e-12
assert np.all(np.isfinite(u_num))
assert len(t_hist) == len(U_hist)

u_ex = exact_solution(x, t_final, alpha=alpha)
err_l2 = np.sqrt(np.mean((u_num - u_ex)**2))
err_inf = np.max(np.abs(u_num - u_ex))

print(f"Checks OK")
print(f"t_final={t_final:.6f}, r={r:.4f}")
print(f"Error L2   = {err_l2:.3e}")
print(f"Error Linf = {err_inf:.3e}")

## 8) Visualización: comparación en el tiempo final

In [ ]:
u_ex = exact_solution(x, t_final, alpha=alpha)

plt.figure(figsize=(8,4))
plt.plot(x, u_num, 'o-', ms=3, label='Numérica (FTCS)')
plt.plot(x, u_ex, '--', lw=2, label='Exacta')
plt.xlabel('x')
plt.ylabel('u(x,T)')
plt.title('Comparación en el tiempo final')
plt.grid(alpha=0.25)
plt.legend()
plt.show()

## 9) Visualización: evolución temporal en el cuaderno

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
line_num, = ax.plot([], [], lw=2, label='FTCS')
line_ex, = ax.plot([], [], '--', lw=1.8, label='Exacta')
txt = ax.text(0.02, 0.92, '', transform=ax.transAxes)

ax.set_xlim(0, 1)
ax.set_ylim(-0.05, 1.05)
ax.set_xlabel('x')
ax.set_ylabel('u(x,t)')
ax.set_title('Evolución de la temperatura')
ax.grid(alpha=0.25)
ax.legend()

def init():
    line_num.set_data([], [])
    line_ex.set_data([], [])
    txt.set_text('')
    return line_num, line_ex, txt

def update(k):
    tk = t_hist[k]
    line_num.set_data(x, U_hist[k])
    line_ex.set_data(x, exact_solution(x, tk, alpha=alpha))
    txt.set_text(f't={tk:.4f}')
    return line_num, line_ex, txt

ani = FuncAnimation(fig, update, frames=len(t_hist), init_func=init, interval=80, blit=True)
plt.close(fig)
ani

## 10) Ejercicios propuestos

1. Repite el experimento con $u(x,0)=\sin(2\pi x)$ y compara la tasa de decaimiento.
2. Fija $N$ y aumenta $r$ hasta perder estabilidad; documenta el comportamiento observado.
3. Construye una tabla de error para varios valores de $N$, manteniendo $r$ constante.
4. Usa una combinación de senos como condición inicial e identifica qué modos dominan la dinámica.